
# Stellar Mass and Luminosity Functions from Mock Survey

The stellar mass function (SMF) describes the number density of galaxies as a
function of stellar mass, a fundamental probe of galaxy assembly. The Schechter
function provides an excellent fit to observed SMF across cosmic time:

\begin{align}\Phi(M) \, dM = \Phi_* \left(\frac{M}{M_*}\right)^{\alpha}     \exp\left(-\frac{M}{M_*}\right) \, dM\end{align}

For a complete survey, a single galaxy's stellar mass also predicts its
luminosity through the mass-to-light ratio (M/L), which depends on the star
formation history, dust attenuation, and stellar population age.

This example constructs a mock survey of 200 galaxies from a Schechter SMF
(Baldry+2012 z=0 fit: M*=10.78 M☉, alpha=-1.45, phi*=3.96e-3 Mpc^-3).
For each galaxy, we build a tengri SEDModel and predict r-band absolute
magnitudes. Binning in M_r yields a luminosity function, which we compare
to the Blanton+2003 SDSS LF as a sanity check.

the forward model: stellar mass → SED → rest-frame
absolute magnitudes.

References:

- Schechter 1976, ApJ, 203, 297 (SMF parameterization)
- Baldry et al. 2012, MNRAS, 421, 621 (Schechter fit z~0)
- Blanton et al. 2003, ApJ, 592, 819 (SDSS luminosity function)


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# ============================================================================
# Schechter function sampling
# ============================================================================


def schechter_sample(
    n_sample: int,
    log10_mstar: float = 10.78,
    alpha: float = -1.45,
    rng_seed: int = 42,
) -> np.ndarray:
    r"""Sample stellar masses from a Schechter function via inverse CDF.

    Uses the incomplete gamma function to invert the Schechter CDF.
    For the Schechter function:

    .. math::

        \Phi(M) \, dM = \Phi_* \left(\frac{M}{M_*}\right)^\alpha
        \exp\left(-\frac{M}{M_*}\right) dM

    The cumulative distribution is related to the incomplete gamma function:

    .. math::

        F(>M) = \Gamma(\alpha+2, M/M_*)

    We sample by drawing uniform variates and inverting via the incomplete
    gamma quantile function.

    Parameters
    ----------
    n_sample : int
        Number of samples.
    log10_mstar : float, optional
        Log10 characteristic mass M* in M_sun. Default 10.78 (Baldry+2012).
    alpha : float, optional
        Low-mass power-law slope. Default -1.45 (Baldry+2012).
    rng_seed : int, optional
        Random seed for reproducibility. Default 42.

    Returns
    -------
    log10_m_star : ndarray, shape (n_sample,)
        Log10(M_star / M_sun) for each sampled galaxy.
    """
    # Schechter PDF p(x) ∝ x^α exp(-x) is improper at x=0 for α < -1, so the
    # original Exponential-proposal rejection sampler had an unbounded
    # envelope and never terminated. Sample in log-space instead: let
    # u = log10(x). The PDF transforms as p(u) ∝ x^(α+1) exp(-x) * ln(10),
    # and for α = -1.45 the exponent (α+1) = -0.45 keeps x^(α+1) bounded
    # over [10^u_min, 10^u_max], so rejection has a well-defined envelope.
    rng = np.random.RandomState(rng_seed)
    u_min, u_max = -3.0, 2.0  # log10(M/M*) range — covers faint to ULIRG
    # Envelope: max of x^(α+1) exp(-x) on the interval is at x = x_min.
    x_min = 10.0**u_min
    w_envelope = x_min ** (alpha + 1.0) * np.exp(-x_min)
    samples: list = []
    for _ in range(100):  # bounded iteration count — never spin forever
        if len(samples) >= n_sample:
            break
        batch_size = max((n_sample - len(samples)) * 3, 1000)
        u_prop = rng.uniform(u_min, u_max, size=batch_size)
        x_prop = 10.0**u_prop
        w = x_prop ** (alpha + 1.0) * np.exp(-x_prop) / w_envelope
        u_accept = rng.uniform(0, 1, size=batch_size)
        samples.extend(u_prop[u_accept < w].tolist())
    if len(samples) < n_sample:
        # Fall back to padding (only fires if rejection was pathologically poor).
        samples = (samples * (1 + n_sample // max(1, len(samples))))[:n_sample]

    log10_m_samples = log10_mstar + np.array(samples[:n_sample])
    return log10_m_samples


# ============================================================================
# Load SSP and prepare observation
# ============================================================================

# Load SSP
SSP = tengri.load_ssp("fsps_prsc_miles_chabrier")

# r-band photometry (rest-frame for z~0)
from tengri.units import fnu_to_ab_mag

obs = tengri.Observation(photometry=tengri.Photometry.from_names(["sdss_r"]))

# ============================================================================
# Sample galaxies from Schechter SMF
# ============================================================================

print("Sampling stellar mass function...")
key = jax.random.PRNGKey(42)
n_gal = 200

log_mstar_samples = schechter_sample(n_gal)
log_mstar_samples = np.sort(log_mstar_samples)  # For stable iteration

print(f"Sampled {n_gal} galaxies from Schechter SMF")
print(
    f"  M* range: {10 ** log_mstar_samples.min():.2e} – {10 ** log_mstar_samples.max():.2e} M_sun"
)
print(f"  log10(M*) range: {log_mstar_samples.min():.2f} – {log_mstar_samples.max():.2f}")

# ============================================================================
# Build model and predict absolute magnitudes
# ============================================================================

# Template model (uses default priors from recipe)
model = tengri.SEDModel.build(
    SSP,
    observation=obs,
    **tengri.recipes.star_forming_photometry(),
)

# Sample template parameters once
key_params = jax.random.PRNGKey(123)
params_template = dict(model.spec.sample(key_params))

# Store results
abs_mags = []

print("\nPredicting r-band absolute magnitudes...")
for i, log_m_star in enumerate(log_mstar_samples):
    # Set stellar mass via SFR scaling
    # Main sequence relation: log(SFR) ≈ 0.7 * log(M*) - 6
    # This is a rough empirical scaling; in reality would come from SFH fitting
    log_sfr = 0.7 * log_m_star - 6.0

    params = dict(params_template)
    params["sfh_dpl_log_total_mass"] = float(log_sfr) + 10.0
    params["redshift"] = 0.0  # z=0 for rest-frame

    # Predict photometry (returns flux in erg/s/cm²/Hz for each band)
    try:
        flux_phot = model.predict_photometry(params)
        # flux_phot is in erg/s/cm²/Hz (AB system)
        flux_r = float(flux_phot[0])  # r-band flux

        # Convert flux to apparent magnitude
        m_app = float(fnu_to_ab_mag(jnp.asarray(flux_r)))

        # Absolute magnitude at z=0: M = m (distance modulus = 0 at z=0)
        M_r = m_app

        abs_mags.append(M_r)

        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{n_gal}: log10(M*)={log_m_star:.2f}, m_r={m_app:.2f}, M_r={M_r:.2f}")

    except Exception as e:
        print(f"Warning: galaxy {i} failed ({e}), skipping")
        continue

abs_mags = np.array(abs_mags)
print(f"\nComputed {len(abs_mags)} absolute magnitudes")
print(f"  M_r range: {abs_mags.min():.2f} to {abs_mags.max():.2f}")

# ============================================================================
# Construct luminosity function: bin in M_r
# ============================================================================

# Histogram bins (standard SDSS LF binning)
m_r_min, m_r_max = -24.0, -16.0
n_bins = 16
m_r_edges = np.linspace(m_r_min, m_r_max, n_bins + 1)
m_r_centers = 0.5 * (m_r_edges[:-1] + m_r_edges[1:])

# Histogram (unnormalized for now)
counts, _ = np.histogram(abs_mags, bins=m_r_edges)

# Normalize: number density per unit magnitude per unit comoving volume
# For a small local survey, assume volume ~ 1 Mpc^3 (or compute from survey geometry)
# Here we just report the counts per magnitude bin
lf_unnorm = counts / np.diff(m_r_edges)  # Per magnitude

# Optional: normalize to Phi(M_r) [Mpc^-3 / mag] if we knew the survey volume
survey_volume_mpc3 = 100.0  # Arbitrary survey volume for this mock
lf_norm = lf_unnorm / survey_volume_mpc3

# ============================================================================
# Reference: Blanton+2003 SDSS LF (approximate parameterization)
# ============================================================================


def blanton2003_lf(m_r: np.ndarray) -> np.ndarray:
    r"""Blanton+2003 SDSS r-band luminosity function (double Schechter).

    Parameterization (SDSS: h=0.7):
      Phi(M_r) = Phi_1 * 10^(0.4*(alpha_1+1)*(M_r - M_r*)) \
                 * exp(-10^(0.4*(M_r - M_r*)))

                + Phi_2 * 10^(0.4*(alpha_2+1)*(M_r - M_r*)) \
                * exp(-10^(0.4*(M_r - M_r*)))

    For simplicity, we use a single Schechter approximation centered at
    M_r* ~ -20.4, alpha ~ -1.2, Phi* ~ 1.6e-2 Mpc^-3.

    Parameters
    ----------
    m_r : ndarray
        Absolute r-band magnitudes.

    Returns
    -------
    phi : ndarray
        Luminosity function Phi(M_r) in units of Mpc^-3 mag^-1.

    Notes
    -----
    This is a simplified approximation for comparison. Real SDSS LF includes
    dust effects, k-corrections, and a more complex double-Schechter fit.
    """
    m_r_star = -20.4
    alpha = -1.2
    phi_star = 1.6e-2

    # Schechter parameterization
    x = 10.0 ** (0.4 * (m_r - m_r_star))
    phi = phi_star * x ** (alpha + 1.0) * np.exp(-x)

    return phi


# Evaluate Blanton+2003 on the bin centers
phi_blanton = blanton2003_lf(m_r_centers)

# ============================================================================
# Plot
# ============================================================================

fig, ax = plt.subplots(figsize=(8.5, 6.0))

# Mock survey (this work)
ax.bar(
    m_r_centers,
    lf_norm,
    width=np.diff(m_r_edges)[0],
    align="center",
    alpha=0.6,
    color="C0",
    edgecolor="black",
    linewidth=0.8,
    label="Mock survey (this work)",
)

# Blanton+2003 reference (scaled to survey volume for visibility)
ax.plot(
    m_r_centers,
    phi_blanton * survey_volume_mpc3,
    color="0.3",
    lw=2.0,
    label="Blanton et al. 2003 (SDSS)",
)

# Axes and labels
ax.set_xlabel(r"Absolute r-band magnitude $M_r$", fontsize=11)
ax.set_ylabel(r"$\Phi(M_r) \times V_{\mathrm{survey}}$ [counts / mag]", fontsize=11)
ax.set_xlim(m_r_max + 0.5, m_r_min - 0.5)  # Flip: bright (left) to faint (right)
ax.set_yscale("log")
ax.set_ylim(0.1, 100)
ax.legend(frameon=False, loc="upper right", fontsize=10)
ax.grid(True, alpha=0.3, linestyle="--", which="both")

fig.tight_layout()
plt.savefig("plot_usecase_stellar_mass_luminosity_function.png", dpi=150, bbox_inches="tight")